In [2]:
#Task1
import numpy as np
from torchvision import datasets, transforms
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt


def load_cifar10(n_train=5000):
    """
    Load CIFAR-10 and flatten images.
    Only take first n_train samples to reduce training time.
    """
    transform = transforms.ToTensor()
    train_dataset = datasets.CIFAR10(root='./cifar_data', train=True,
                                     download=True, transform=transform)
    test_dataset = datasets.CIFAR10(root='./cifar_data', train=False,
                                    download=True, transform=transform)

    X_train = np.stack([train_dataset[i][0].numpy().reshape(-1) for i in range(n_train)], axis=0)
    y_train = np.array([train_dataset[i][1] for i in range(n_train)])

    X_test = np.stack([img.numpy().reshape(-1) for img, _ in test_dataset], axis=0)
    y_test = np.array([label for _, label in test_dataset])

    return X_train, y_train, X_test, y_test, train_dataset.classes


def compute_pca_variance(X_train_std):
    """
    Fit PCA to compute cumulative variance ratio.
    """
    pca_full = PCA().fit(X_train_std)
    cum_ratio = np.cumsum(pca_full.explained_variance_ratio_)
    return cum_ratio


def get_n_components(cum_ratio, target_ratio):
    """
    Find smallest k such that >= target_ratio explained variance.
    """
    return np.searchsorted(cum_ratio, target_ratio) + 1


def apply_pca(X_train_std, X_test_std, n_components):
    """
    Apply PCA with n_components.
    """
    pca = PCA(n_components=n_components)
    X_train_pca = pca.fit_transform(X_train_std)
    X_test_pca = pca.transform(X_test_std)
    return X_train_pca, X_test_pca, pca


if __name__ == "__main__":
    print("Loading CIFAR-10...")
    X_train, y_train, X_test, y_test, class_names = load_cifar10()

    print("Standardizing data...")
    scaler = StandardScaler()
    X_train_std = scaler.fit_transform(X_train)
    X_test_std = scaler.transform(X_test)

    print("Computing PCA variance ratios...")
    cum_ratio = compute_pca_variance(X_train_std)

    # Calculate dimensions for each ratio
    ratios = [0.10, 0.30, 0.50, 0.70]
    dims = {f"{int(r*100)}%": get_n_components(cum_ratio, r) for r in ratios}
    dims["100%"] = X_train_std.shape[1]

    print("\n=== PCA Dimension Summary ===")
    for k, v in dims.items():
        print(f"Retain {k} variance → {v} dimensions")

    # Apply PCA and save data
    for name, ratio in zip(["10%", "30%", "50%", "70%"], ratios):
        d = dims[name]
        print(f"\nApplying PCA for {name} ({d} dims)...")
        Xtr_pca, Xte_pca, pca_model = apply_pca(X_train_std, X_test_std, d)

        np.save(f"X_train_pca_{name}.npy", Xtr_pca)
        np.save(f"X_test_pca_{name}.npy", Xte_pca)

    # Raw standardized
    np.save("X_train_raw.npy", X_train_std)
    np.save("X_test_raw.npy", X_test_std)
    np.save("y_train.npy", y_train)
    np.save("y_test.npy", y_test)

    print("\nSaved all PCA features and labels ✔")

Loading CIFAR-10...
Standardizing data...
Computing PCA variance ratios...

=== PCA Dimension Summary ===
Retain 10% variance → 1 dimensions
Retain 30% variance → 2 dimensions
Retain 50% variance → 5 dimensions
Retain 70% variance → 15 dimensions
Retain 100% variance → 3072 dimensions

Applying PCA for 10% (1 dims)...

Applying PCA for 30% (2 dims)...

Applying PCA for 50% (5 dims)...

Applying PCA for 70% (15 dims)...

Saved all PCA features and labels ✔


In [4]:
#Task2
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score


def load_features():
    """
    Load PCA features saved from Task1.
    """
    X_train_raw = np.load("X_train_raw.npy")
    X_test_raw = np.load("X_test_raw.npy")

    y_train = np.load("y_train.npy")
    y_test = np.load("y_test.npy")

    pcas = {}
    for name in ["10%", "30%", "50%", "70%"]:
        pcas[name] = (
            np.load(f"X_train_pca_{name}.npy"),
            np.load(f"X_test_pca_{name}.npy")
        )

    pcas["100%"] = (X_train_raw, X_test_raw)

    return X_train_raw, X_test_raw, y_train, y_test, pcas


def run_mlp_cv(X, y, hidden=(256,), lr=0.001):
    model = MLPClassifier(
        hidden_layer_sizes=hidden,
        learning_rate_init=lr,
        max_iter=150,
        random_state=42
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    return scores.mean(), scores.std()


def evaluate_test_set(X_train, y_train, X_test, y_test, hidden=(256,), lr=0.001):
    model = MLPClassifier(
        hidden_layer_sizes=hidden,
        learning_rate_init=lr,
        max_iter=120,
        random_state=42
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("Test Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))


if __name__ == "__main__":
    X_raw_tr, X_raw_te, y_train, y_test, pcas = load_features()

    print("\n=== Running MLP on all PCA feature sets ===")
    results = []

    for name, (Xtr, Xte) in pcas.items():
        print(f"\nEvaluating MLP on PCA {name} ({Xtr.shape[1]} dims)...")
        mean_acc, std_acc = run_mlp_cv(Xtr, y_train)
        results.append((name, Xtr.shape[1], mean_acc, std_acc))
        print(f"CV Accuracy = {mean_acc:.4f} ± {std_acc:.4f}")

    print("\n=== Summary Table ===")
    for name, dim, acc, std in results:
        print(f"{name}: {dim} dims → {acc:.4f} ± {std:.4f}")

    # Example: choose PCA-50% to test final performance
    print("\n=== Test-set performance (example using PCA-50%) ===")
    Xtr, Xte = pcas["50%"]
    evaluate_test_set(Xtr, y_train, Xte, y_test)


=== Running MLP on all PCA feature sets ===

Evaluating MLP on PCA 10% (1 dims)...
CV Accuracy = 0.1634 ± 0.0125

Evaluating MLP on PCA 30% (2 dims)...
CV Accuracy = 0.2136 ± 0.0201

Evaluating MLP on PCA 50% (5 dims)...


/opt/anaconda3/envs/MachineLearning/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/MachineLearning/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(


CV Accuracy = 0.2802 ± 0.0232

Evaluating MLP on PCA 70% (15 dims)...


/opt/anaconda3/envs/MachineLearning/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/MachineLearning/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/MachineLearning/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/envs/MachineLearning/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
 

CV Accuracy = 0.3480 ± 0.0093

Evaluating MLP on PCA 100% (3072 dims)...
CV Accuracy = 0.3990 ± 0.0173

=== Summary Table ===
10%: 1 dims → 0.1634 ± 0.0125
30%: 2 dims → 0.2136 ± 0.0201
50%: 5 dims → 0.2802 ± 0.0232
70%: 15 dims → 0.3480 ± 0.0093
100%: 3072 dims → 0.3990 ± 0.0173

=== Test-set performance (example using PCA-50%) ===
Test Accuracy: 0.2808
              precision    recall  f1-score   support

           0       0.38      0.47      0.42      1000
           1       0.25      0.09      0.14      1000
           2       0.23      0.10      0.14      1000
           3       0.21      0.14      0.16      1000
           4       0.25      0.31      0.28      1000
           5       0.23      0.41      0.30      1000
           6       0.30      0.28      0.29      1000
           7       0.23      0.26      0.25      1000
           8       0.37      0.52      0.43      1000
           9       0.28      0.23      0.25      1000

    accuracy                           0.28    

/opt/anaconda3/envs/MachineLearning/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (120) reached and the optimization hasn't converged yet.
  warnings.warn(


In [6]:
#Task3A-Random Forest
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score


def load_features():
    X_train_raw = np.load("X_train_raw.npy")
    X_test_raw = np.load("X_test_raw.npy")
    y_train = np.load("y_train.npy")
    y_test = np.load("y_test.npy")

    pcas = {}
    for name in ["10%", "30%", "50%", "70%"]:
        pcas[name] = (
            np.load(f"X_train_pca_{name}.npy"),
            np.load(f"X_test_pca_{name}.npy")
        )
    pcas["100%"] = (X_train_raw, X_test_raw)

    return X_train_raw, X_test_raw, y_train, y_test, pcas


def run_rf_cv(X, y, n_estimators=100, max_depth=None):
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    acc = cross_val_score(model, X, y, scoring="accuracy", cv=cv, n_jobs=-1)
    return acc.mean(), acc.std()


def evaluate_test(X_train, y_train, X_test, y_test, n_estimators=100, max_depth=None):
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("Test Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))


if __name__ == "__main__":
    X_raw_tr, X_raw_te, y_train, y_test, pcas = load_features()

    print("\n=== Random Forest Evaluation ===")
    for name, (Xtr, Xte) in pcas.items():
        print(f"\nPCA {name} ({Xtr.shape[1]} dims)")
        mean_acc, std = run_rf_cv(Xtr, y_train)
        print(f"CV Accuracy = {mean_acc:.4f} ± {std:.4f}")

    print("\n--- Test-set Example on PCA-50% ---")
    Xtr, Xte = pcas["50%"]
    evaluate_test(Xtr, y_train, Xte, y_test)


=== Random Forest Evaluation ===

PCA 10% (1 dims)
CV Accuracy = 0.1100 ± 0.0025

PCA 30% (2 dims)
CV Accuracy = 0.1624 ± 0.0069

PCA 50% (5 dims)
CV Accuracy = 0.2608 ± 0.0072

PCA 70% (15 dims)
CV Accuracy = 0.3666 ± 0.0125

PCA 100% (3072 dims)
CV Accuracy = 0.3944 ± 0.0103

--- Test-set Example on PCA-50% ---
Test Accuracy: 0.2615
              precision    recall  f1-score   support

           0       0.37      0.42      0.39      1000
           1       0.21      0.14      0.17      1000
           2       0.19      0.21      0.20      1000
           3       0.19      0.15      0.17      1000
           4       0.21      0.22      0.21      1000
           5       0.25      0.23      0.24      1000
           6       0.25      0.29      0.27      1000
           7       0.22      0.17      0.19      1000
           8       0.36      0.46      0.40      1000
           9       0.29      0.32      0.30      1000

    accuracy                           0.26     10000
   macro avg

In [14]:
#Task3B-KNN
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score


def load_features():
    X_train_raw = np.load("X_train_raw.npy")
    X_test_raw = np.load("X_test_raw.npy")
    y_train = np.load("y_train.npy")
    y_test = np.load("y_test.npy")

    pcas = {}
    for name in ["10%", "30%", "50%", "70%"]:
        pcas[name] = (
            np.load(f"X_train_pca_{name}.npy"),
            np.load(f"X_test_pca_{name}.npy")
        )
    pcas["100%"] = (X_train_raw, X_test_raw)

    return X_train_raw, X_test_raw, y_train, y_test, pcas


def run_knn_cv(X, y, k=5):
    model = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    acc = cross_val_score(model, X, y, scoring="accuracy", cv=cv, n_jobs=-1)
    return acc.mean(), acc.std()


def evaluate_test(X_train, y_train, X_test, y_test, k=5):
    model = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("Test Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))


if __name__ == "__main__":
    _, _, y_train, y_test, pcas = load_features()

    print("\n=== KNN Evaluation ===")
    for name, (Xtr, Xte) in pcas.items():
        print(f"\nPCA {name} ({Xtr.shape[1]} dims)")
        mean_acc, std = run_knn_cv(Xtr, y_train)
        print(f"CV Accuracy = {mean_acc:.4f} ± {std:.4f}")

    print("\n--- Test-set Example on PCA-50% ---")
    Xtr, Xte = pcas["50%"]
    evaluate_test(Xtr, y_train, Xte, y_test)


=== KNN Evaluation ===

PCA 10% (1 dims)
CV Accuracy = 0.1162 ± 0.0072

PCA 30% (2 dims)
CV Accuracy = 0.1618 ± 0.0084

PCA 50% (5 dims)
CV Accuracy = 0.2332 ± 0.0115

PCA 70% (15 dims)
CV Accuracy = 0.2974 ± 0.0090

PCA 100% (3072 dims)
CV Accuracy = 0.2644 ± 0.0137

--- Test-set Example on PCA-50% ---
Test Accuracy: 0.2219
              precision    recall  f1-score   support

           0       0.26      0.47      0.33      1000
           1       0.17      0.20      0.18      1000
           2       0.15      0.19      0.17      1000
           3       0.18      0.15      0.17      1000
           4       0.20      0.18      0.19      1000
           5       0.22      0.16      0.18      1000
           6       0.21      0.19      0.20      1000
           7       0.22      0.15      0.17      1000
           8       0.33      0.32      0.32      1000
           9       0.29      0.21      0.25      1000

    accuracy                           0.22     10000
   macro avg       0.2